In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter # Import the rate limiter

load_dotenv()

# Get API key from .env
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Make sure it's set in the .env file.")

# Initialize a rate limiter for 10 requests per minute
# This is the crucial addition to manage your API calls
rate_limiter = InMemoryRateLimiter(requests_per_second=10 / 60)

# Define the LLM with the rate limiter included
llm_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0,
    max_retries=3,
    rate_limiter=rate_limiter,
)

In [2]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

# Create the agent

model = llm_model
search = TavilySearch(max_results=2)
tools = [search]
agent_executor = create_agent(model, tools)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

system_message = ("""You are a web search agent."""
)

agent_executor = create_agent(
    model=llm_model,
    tools=[search],
)

task_input = (
    "Find the latest news about how solo entrepreneur use AI to become multimillionaires. "
)

response = agent_executor.invoke({"messages": [("user", task_input)]})

print(response["messages"][-1].content)

[{'type': 'text', 'text': "There are a growing number of solo entrepreneurs using AI to achieve significant wealth. In the last three months alone, 11 individuals under 30 have become ultra-wealthy thanks to AI innovations. Examples include Polymarket CEO Shayne Coplan, Fabian Hedin (cofounder of vibe coding startup Loveable), and AI entrepreneur Arvid Lunnemark. Additionally, Luana Lopes Lara, the world's youngest female self-made billionaire, saw her fortune reach $1.3 billion after her prediction market startup, Kalshi, achieved an $11 billion valuation, also leveraging AI.", 'extras': {'signature': 'Cv8FAXLI2nztSbEJ0SI2DlF7d9K/W9bDTPbsRD0IjuIJG8U+WRgyUXNyf327JAL1pCh4/ULUCPREVJPapftcfkkqutXlQZ94l94f4es52P/9cAnpcClyUuNXUvbsaLJC05keWo47N9fLE7dlTyfiKhw8jIkilYN8amAIfOsCM6RTIWeVOwt/JtJE7m5pPuVzlyqLXJwXWIzFTXee21Xm2im5o5B69TM3kr3a+hBT9qQVDP+7MjfLC44ACQ2Rvf6afdLUclC1LiBMQLIYhO7vUmmfkqsuzzGhzrQYLI3ikdt74de5p/75Siq0PpqiQbBPfqD0QHOujmt1fxI6eB+aHaqPz7Wj+RCcvuwv0RR1p3/PjjQE08xl1TZrNLoTT97smrCQG

In [4]:
from gtts import gTTS

def clean_text(x):
    if isinstance(x, list):
        return " ".join(str(i) for i in x)
    return str(x)

def generate_tts_audio(text, filename="output_audio.mp3"):
    text = clean_text(text)

    tts = gTTS(text=text, lang='en')
    tts.save(filename)

    print(f"Audio saved as {filename}")
    return filename


In [5]:
raw_output = response["messages"][-1].content
text = clean_text(raw_output)
generate_tts_audio(text, "industry_ears_news.mp3")

Audio saved as industry_ears_news.mp3


'industry_ears_news.mp3'